# ScoreIA - Descricao Tecnica do Agente

Documento vivo com o estado implementado do agente ScoreIA.

Snapshot tecnico atualizado em: `2026-03-26`.


## 1. Visao geral

O ScoreIA hoje tem tres superficies principais de execucao:

- `main.py`: CLI principal e roteador de comandos do agente;
- `tools/ml/webapp/app.py`: interface web do pipeline de ML;
- `tools/img_gen/webapp/app.py`: interface web do pipeline de imagem.

Nucleos funcionais:

- `core/agent.py`: orquestracao de LLM, estudo, planejamento e pipeline de ML;
- `tools/ml/`: modelagem, feature engineering, otimizacao, relatorio e exportacao;
- `tools/img_gen/`: projetos visuais, registry de modelos, runtime de geracao e webapp da fase 6.


## 2. Organizacao atual do repositorio

- `main.py`: entrada CLI
- `core/`: memoria, agente e orquestracao conversacional
- `tools/ml/`: pipeline de ML
- `tools/img_gen/`: pipeline de imagem
- `projects/img_gen/`: estado persistente por projeto de imagem
- `artifacts/ml/`: artefatos de runs de ML
- `artifacts/img_gen/`: artefatos de runs de imagem
- `models/img_gen/`: pesos e registry local de modelos
- `docs/`: documentacao funcional e tecnica


## 3. Funcionalidades ativas

### Agente e CLI

- estudo guiado com referencias externas quando habilitadas;
- planejamento pessoal e manutencao de tarefas;
- abertura das interfaces web de ML e imagem a partir do CLI.

### Pipeline de ML

- pipeline supervisionado com `train`, `test` e `oot`;
- optuna/CV para `lightgbm`, `mlp` e `auto`;
- multiestagio com `model_full`, `model_prunned` e `model_reduced` quando aplicavel;
- validacao temporal e persistencia de dataset na webapp;
- relatorio tecnico HTML, scored dataset e pacote exportavel.

### Pipeline de imagem

- `txt2img`, `img2img` e `inpainting` com `diffusers`;
- suporte a projetos, `visual_bible`, referencias, imports, `working_set` e `promoted_assets`;
- webapp simplificada para fluxo livre ou por projeto;
- registry local de modelos por alias;
- manifests por run com parametros, modelo, seed e outputs.


## 4. Funcoes principais da CLI (`main.py`)

### `handle_interacao(agent, mensagem)`

Uso: roteia um comando textual do usuario para o subsistema correto.

Funcionamento:

- reconhece comandos como `estudar:`, `planejar:`, `modelagem` e `gerar imagem`;
- chama o metodo adequado do `Agente`;
- registra a interacao na memoria e persiste o estado.

Parametros principais:

- `agent`: instancia de `Agente`;
- `mensagem`: texto cru digitado no CLI.

### `_abrir_interface_modelagem()`

Uso: sobe a webapp de ML em thread daemon e abre o navegador.

Parametros: nenhum explicito; usa `SCOREIA_WEBAPP_HOST` e `SCOREIA_WEBAPP_PORT`.

### `_abrir_interface_imagem()`

Uso: sobe a webapp de imagem em thread daemon e abre o navegador.

Parametros: nenhum explicito; usa `SCOREIA_IMG_WEBAPP_HOST` e `SCOREIA_IMG_WEBAPP_PORT`.

### `run_cli()`

Uso: loop principal do terminal local do ScoreIA.


## 5. Funcoes principais do agente (`core/agent.py`)

### `Agente.executar(mensagem, tipo=None, tema=None)`

Uso: entrada geral para interacoes livres com o LLM.

Funcionamento:

- resolve contexto recente da memoria;
- monta o prompt base governado por `memory/profile.json`;
- chama o modelo local via Ollama;
- registra a interacao na memoria episodica.

Parametros:

- `mensagem`: pedido atual do usuario;
- `tipo`: classificador opcional da interacao;
- `tema`: tema opcional para indexacao da memoria.

### `Agente.estudar(tema)`

Uso: monta um bloco de estudo tecnico com explicacao, exercicios e referencias.

Parametro principal: `tema`.

### `Agente.planejar(objetivo, horizonte="semanal")`

Uso: gera um plano de organizacao pessoal baseado em memoria e preferencias.

Parametros principais:

- `objetivo`: meta do planejamento;
- `horizonte`: janela temporal desejada.

### `Agente.analisar_dados(config)`

Uso: orquestra a execucao ponta a ponta do pipeline de ML.

Funcionamento resumido:

- inicializa contexto de run;
- carrega e particiona o dataset;
- aplica feature selection temporal e pos-processamento de features;
- executa os estagios `full`, `prunned` e `reduced` quando cabiveis;
- consolida retorno, artefatos e relatorios.

Parametro principal: `config`, dicionario com caminho do dataset, target, modelo, opcoes temporais, pruning, reducao e balanceamento.


## 6. Funcoes principais do pipeline de ML

### `tools/ml/webapp/app.py:index()`

Uso: endpoint principal da interface web de ML.

Funcionamento:

- diferencia `validate_only` de `train_pipeline`;
- persiste o dataset validado para reuso posterior;
- valida feature temporal e limites de folds antes do treino;
- chama `Agente.analisar_dados(config)` no treino completo.

Parametros de formulario mais importantes:

- `file` ou `persisted_dataset_path`
- `target`
- `task`
- `modelo`
- `temporal_feature`
- `oot_months`
- `stability_feature_selection`
- `classification_balance_strategy`

### `carregar_pipeline_e_particionar(config)`

Uso: constroi o `DataPipeline`, carrega os dados e produz os splits `train/test/oot`.

Parametro principal: `config`, com definicoes de caminho, target, task, temporalidade e segmentacao.

### `treinar_modelo_final_ml(...)`

Uso: executa o treino final do estagio de modelagem ja com o conjunto de features definido.

Parametros principais: dados transformados, target, configuracao do estagio, metadata de features e opcoes do modelo.

### `otimizar_modelo_cv(...)`

Uso: roda Optuna + validacao cruzada para `lightgbm`, `mlp` ou selecao `auto`.

Parametros principais:

- matriz de features e target;
- `modelo`
- `task`
- `n_trials`
- `early_stopping_rounds`
- configuracoes de folds e metricas.


## 7. Funcoes principais do runtime de imagem (`tools/img_gen/image_generation.py`)

### `generate_txt2img(...)`

Uso: gera imagens novas a partir de texto.

Funcionamento:

- valida prompt e dimensoes;
- prepara runtime e checkpoint por alias;
- aplica preset automatico de qualidade para SD 1.5;
- troca o scheduler para `DPM++ 2M Karras` quando suportado;
- opcionalmente carrega `IP-Adapter` e `ControlNet`;
- executa o pipeline `DiffusionPipeline` ou `StableDiffusionControlNetPipeline`;
- salva imagens e manifests da run.

Parametros principais:

- `prompt`
- `negative_prompt`
- `model_alias`
- `width`, `height`
- `num_inference_steps`
- `guidance_scale`
- `num_images`
- `project_id`
- `run_id`

Parametros opcionais avancados:

- `ip_adapter_alias`, `ip_adapter_image_path`, `ip_adapter_scale`
- `controlnet_alias`, `control_image_path`, `controlnet_conditioning_scale`
- `safety_checker_enabled`
- `quality_preset`
- `seed`

### `generate_img2img(...)`

Uso: refina ou transforma uma imagem-base existente.

Funcionamento:

- carrega a imagem-base;
- aplica preset automatico de qualidade para SD 1.5;
- troca o scheduler para `DPM++ 2M Karras` quando suportado;
- usa `AutoPipelineForImage2Image` ou variante com `ControlNet`;
- aplica `strength` para controlar o quanto a imagem original sera preservada;
- salva outputs e snapshots dos inputs usados.

Parametros principais:

- `prompt`
- `init_image_path`
- `strength`
- `model_alias`
- `num_inference_steps`
- `guidance_scale`
- `num_images`
- `quality_preset`
- `project_id`

### `generate_inpainting(...)`

Uso: altera apenas a area definida por uma mascara.

Funcionamento:

- valida `init_image_path` e `mask_image_path`;
- aplica preset automatico de qualidade para SD 1.5;
- troca o scheduler para `DPM++ 2M Karras` quando suportado;
- verifica se o checkpoint suporta `inpainting`;
- usa `AutoPipelineForInpainting`;
- aplica blur controlado na mascara para melhorar a integracao;
- salva outputs e snapshots da imagem-base e da mascara.

Parametros principais:

- `prompt`
- `init_image_path`
- `mask_image_path`
- `strength`
- `model_alias`
- `num_inference_steps`
- `guidance_scale`
- `num_images`
- `quality_preset`
- `mask_blur_radius`
- `project_id`


## 8. Funcoes principais de projeto (`tools/img_gen/project_registry.py`)

### `create_project(...)`

Uso: cria um novo projeto de imagem e inicializa sua estrutura de pastas e manifests.

Parametros principais:

- `name`
- `project_type`
- `description`
- `tags`
- `project_id` opcional
- `metadata`

### `add_visual_bible_version(...)`

Uso: registra uma nova versao de `visual_bible`.

Parametros principais:

- `project_id`
- `label`
- `summary`
- `prompt_anchor`
- `negative_prompt_anchor`
- `style_profile`
- `characters`
- `environments`
- `composition_rules`
- `continuity_rules`
- `set_active`

### `update_visual_bible_version(...)`

Uso: edita uma versao existente sem trocar o `version_id`.

Parametros principais: mesmos do `add_visual_bible_version`, com o adicional `version_id`.

### `add_reference_asset(...)`

Uso: registra uma referencia do projeto.

Parametros principais:

- `project_id`
- `source_type`
- `role`
- `caption`
- `relative_path`
- `tags`
- `metadata`

### `add_imported_project(...)`

Uso: registra a importacao parcial de contexto de outro projeto.

Parametros principais:

- `project_id`
- `source_project_id`
- `selection_mode`
- `selectors`
- `selected_reference_ids`
- `selected_promoted_asset_ids`
- `snapshot_policy`

### `add_working_set(...)`

Uso: cria um subconjunto ativo de referencias, assets importados e contexto derivado.

Parametros principais: `label`, `reference_ids`, `promoted_asset_ids`, `imported_project_ids`, `derived_context_ids`, `set_active`.

### `promote_generated_image(...)`

Uso: copia uma imagem gerada para dentro do projeto e a registra como referencia e/ou `promoted_asset`.

Parametros principais:

- `project_id`
- `source_image_path`
- `caption`
- `role`
- `source_run_id`
- `tags`
- `create_reference`
- `create_promoted_asset`

### `set_active_visual_bible(...)` e `set_active_working_set(...)`

Uso: marcam a versao ou conjunto ativo que deve orientar as proximas geracoes.

### `load_project_bundle(project_id)`

Uso: carrega `project.json`, manifests e ponteiros ativos em um unico payload pronto para UI e composicao de prompt.


## 9. Funcoes principais da webapp de imagem (`tools/img_gen/webapp/app.py`)

### `index()`

Uso: endpoint principal da interface de imagem.

Funcionamento:

- resolve abas ativas;
- abre ou cria projetos;
- registra modelos locais;
- recebe uploads;
- chama `generate_txt2img`, `generate_img2img` e `generate_inpainting`;
- renderiza contexto, resultados e mensagens de sucesso/erro.

### `_compose_generation_prompts(bundle, prompt, negative_prompt, include_visual_bible_context)`

Uso: monta o prompt final da execucao.

Funcionamento:

- usa o prompt digitado pelo usuario como base;
- opcionalmente acrescenta o contexto do `visual_bible` ativo;
- devolve `prompt` e `negative_prompt` prontos para o runtime.

### `_build_visual_bible_prompt_context(visual_bible, base_prompt)`

Uso: transforma `visual_bible` em fragmentos de prompt e negative prompt.

Inclui:

- resumo;
- anchors;
- `style_profile`;
- personagens e ambientes relevantes ao texto;
- regras de composicao e continuidade.

### `_save_uploaded_reference_file(project_id, uploaded_file, source_type)`

Uso: salva um upload de referencia direto no projeto, escolhendo a pasta correta por tipo.

### `_save_uploaded_runtime_image(project_id, uploaded_file)`

Uso: salva uploads temporarios usados em `Refino` e `Inpainting` em `artifacts/.../web_uploads`.

### `_build_import_source_image_candidates(source_bundle)`

Uso: monta a galeria visual de imagens importaveis de outro projeto.

### `serve_storage_file(storage, relative_path)`

Uso: serve arquivos de `projects/` e `artifacts/` com validacao de path segura para previews da webapp.


## 10. Contrato de saida atual

### ML

- `artifacts/ml/run_<timestamp>/model_full/...`
- `artifacts/ml/run_<timestamp>/model_prunned/...` quando houver pruning efetivo
- `artifacts/ml/run_<timestamp>/model_reduced/...` quando houver reducao efetiva

### Imagem

- `artifacts/img_gen/_ad_hoc/run_<timestamp>/...` para geracao livre
- `artifacts/img_gen/<project_id>/run_<timestamp>/...` para geracao por projeto
- cada run de imagem grava pelo menos:
  - `generated/`
  - `inputs/`
  - `manifests/generation_manifest.json`
  - manifesto especifico do modo (`txt2img`, `img2img` ou `inpainting`)

### Projetos de imagem

- `projects/img_gen/<project_id>/project.json`
- manifests de `visual_bibles`, `references`, `imported_projects`, `promoted_assets`, `derived_context` e `working_sets`.
